# Opdracht 5 · Random forest
### Classificatie-training · Antwoordmodel-versie

Random forests combineren veel bomen voor een robuuster model. Je gaat:

1. Een **random forest** trainen
2. Het **vergelijken** met logistische regressie en de losse boom uit opdracht 4
3. De **feature importance** bekijken
4. Een **seaborn bar plot** van de importances maken
5. Experimenteren met **`n_estimators`**


## 0. Voorbereiding

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")
print("Bibliotheken geladen.")

In [ ]:
# Data laden en splitsen
data = load_iris(as_frame=True)
df = data.frame.copy()
df["soort"] = df["target"].map({0:"setosa",1:"versicolor",2:"virginica"})
y = df["soort"]; X = df[data.feature_names]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Twee referentiemodellen (uit eerdere opdrachten)
log_model = LogisticRegression(max_iter=2000).fit(X_train, y_train)
tree_model = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print("Referentiemodellen klaar (logistisch en decision tree).")

## 1. Een random forest trainen

### TODO 1 — Train een random forest
Maak een `RandomForestClassifier` met `n_estimators=100` en `random_state=42`,
train op de train-set, en bereken de accuracy.

In [ ]:
# ANTWOORD
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
acc_forest = accuracy_score(y_test, forest.predict(X_test))
print(f"Random forest accuracy: {acc_forest:.3f}")

## 2. Drie modellen vergelijken

In [ ]:
# Vergelijking (al ingevuld)
scores = pd.Series({
    "Logistisch": accuracy_score(y_test, log_model.predict(X_test)),
    "Decision tree": accuracy_score(y_test, tree_model.predict(X_test)),
    "Random forest": acc_forest,
})
print(scores.round(3))

plt.figure(figsize=(6, 4))
scores.plot(kind="bar", color=["#9199B9", "#D97733", "#4473C5"])
plt.ylabel("Accuracy op de test-set"); plt.title("Modelvergelijking")
plt.xticks(rotation=0); plt.ylim(0, 1.05)
plt.axhline(1/3, color="black", linewidth=0.8, linestyle="--", label="baseline (1/3)")
plt.legend(); plt.show()

> **Bespreek:** op Iris presteren alle drie modellen sterk. De verschillen zijn klein. Op
> grotere, complexere datasets (zoals de capstone straks) worden de verschillen groter.

## 3. Feature importance

### TODO 2 — Bekijk de feature importance van het random forest
Maak een pandas Series van `forest.feature_importances_` met `data.feature_names` als index,
gesorteerd aflopend.

In [ ]:
# ANTWOORD
importance = pd.Series(forest.feature_importances_, index=data.feature_names)
importance = importance.sort_values(ascending=False)
print(importance.round(3))

## 4. Bar plot met seaborn

### TODO 3 — Maak een seaborn bar plot van de feature importances
Gebruik `sns.barplot` met `x=importance.values` en `y=importance.index`.

> Tip: een horizontale bar plot leest beter met lange feature-namen.

In [ ]:
# ANTWOORD
plt.figure(figsize=(6, 4))
sns.barplot(x=importance.values, y=importance.index, color="#4473C5", edgecolor="#203864")
plt.xlabel("Belang"); plt.title("Feature importance (random forest)")
plt.show()

## 5. Experimenteren met n_estimators
Hoe verandert de prestatie met het aantal bomen?

### TODO 4 — Train forests met verschillende n_estimators
Voor elke waarde in `aantallen`, train een random forest en bereken de test-accuracy.

In [ ]:
# ANTWOORD
aantallen = [10, 50, 100, 200, 500]
acc_per_n = []
for n in aantallen:
    f = RandomForestClassifier(n_estimators=n, random_state=42).fit(X_train, y_train)
    acc_per_n.append(accuracy_score(y_test, f.predict(X_test)))

for n, a in zip(aantallen, acc_per_n):
    print(f"  n_estimators={n:>4}  -> accuracy {a:.3f}")

## 6. Bespreking
Bespreek met je buur:

- Welk model wint op de test-set? Met hoeveel?
- Welke features dragen het meest bij volgens het random forest? Komt dat overeen met de pairplot?
- Wat gebeurt er met de accuracy als je het aantal bomen verhoogt? Heeft één boom méér zin
  dan 100 bomen?
- Wat kost het random forest aan interpreteerbaarheid t.o.v. de losse boom uit opdracht 4?

---
### Toelichting voor de trainer
- Op Iris presteren alle drie modellen sterk (>0.90). Random forest is iets stabieler, maar het
  verschil is klein op een gemakkelijke dataset. In de capstone wordt het verschil groter.
- **Feature importance**: bloemblad-lengte en -breedte dominant (samen ~85%), kelkblad-breedte
  bijna verwaarloosbaar. Bevestigt wat de pairplot al liet zien.
- **n_estimators**: meer bomen geeft initieel verbetering, daarna plateau. Voorbij ~100 bomen weinig
  winst maar wel langere training-tijd. Goede aanleiding om kosten/baten te bespreken.